<a href="https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/05_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Feature engineering and frozen splits
**Goal:** build one row per speech phone with features and the target (`dur_f`, frames), and fix the train/val/test split **once**.

**Design decisions (put these in the report)**
- **Unit:** one row per speech phone. Silence/`spn` rows are *not* modeled, but we keep two signals about them: `pause_before`, `pause_after`.
- **Features, in five groups** (see `src/features/feature_sets.py`): *identity* (phone, stress, class), *context* (two neighbours each side), *word* (position, length), *utterance* (position, length), *pause/punctuation*.
- **Split by utterance, never by phone:** neighbouring phones of the same sentence are highly correlated; putting some in train and some in test would leak the answer. The split is made over **all 13,100 ids** with a fixed seed so the test set never changes, even if alignment exclusions change later. Phase 2 must use the same files.
- **Target** stays in frames. Models in notebook 06/07 predict `log(frames)`.

**Reads:** alignments, QA, `transcripts_clean.csv`. **Writes:** `data/splits/{train,val,test}_ids.txt`, `data/processed/features.csv.gz`.

In [1]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [2]:
# ---- Write shared module: feature column definitions
import os, importlib
_src = r'''"""Feature column definitions shared by notebooks 05-08 (and Phase 2 if you reuse the features)."""
CAT_COLS = ['base', 'stress_c', 'cls', 'prev1', 'prev2', 'next1', 'next2', 'word_punct']
NUM_COLS = ['pos_in_word', 'word_len', 'rel_pos_in_word', 'is_word_first', 'is_word_last',
            'word_idx', 'n_words', 'rel_pos_utt', 'seq_idx', 'n_phones', 'is_utt_final', 'is_utt_first',
            'pause_after', 'pause_before', 'is_vowel']
GROUPS = {
    'identity': ['base', 'stress_c', 'cls', 'is_vowel'],
    'context': ['prev1', 'prev2', 'next1', 'next2'],
    'word': ['pos_in_word', 'word_len', 'rel_pos_in_word', 'is_word_first', 'is_word_last'],
    'utterance': ['word_idx', 'n_words', 'rel_pos_utt', 'seq_idx', 'n_phones', 'is_utt_final', 'is_utt_first'],
    'pause_punct': ['pause_after', 'pause_before', 'word_punct'],
}
TARGET = 'dur_f'
'''
_p = os.path.join(PROJECT_ROOT, "src/features/feature_sets.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/features/feature_sets.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/features/feature_sets.py


In [3]:
import numpy as np, pandas as pd
from src.features.phones import *
from src.features.feature_sets import *
tr = pd.read_csv(os.path.join(PROCESSED, 'transcripts_clean.csv'))
al = pd.read_csv(os.path.join(ALIGN, 'phone_durations.csv.gz'))
qa = pd.read_csv(os.path.join(ALIGN, 'alignment_qa.csv'))
print('aligned utterances:', al.utt_id.nunique(), '| excluded by QA:', int(qa.exclude.sum()))

aligned utterances: 13100 | excluded by QA: 1684


## Step 1: Frozen split over all ids (seed from `configs/data.yaml`)

In [4]:
sc = cfg['split']; ids = np.array(sorted(tr['id']))
perm = np.random.RandomState(sc['seed']).permutation(len(ids)); ids = ids[perm]
n_tr, n_va = int(len(ids) * sc['train']), int(len(ids) * sc['val'])
split_ids = {'train': ids[:n_tr], 'val': ids[n_tr:n_tr + n_va], 'test': ids[n_tr + n_va:]}
for k, v in split_ids.items():
    path = os.path.join(SPLITS, f'{k}_ids.txt')
    if os.path.exists(path):                                   # never regenerate once created
        v = np.array(open(path).read().split()); split_ids[k] = v
    else:
        open(path, 'w').write('\n'.join(sorted(v)))
sets = {k: set(v) for k, v in split_ids.items()}
assert not (sets['train'] & sets['val']) and not (sets['train'] & sets['test']) and not (sets['val'] & sets['test'])
print({k: len(v) for k, v in split_ids.items()}, '-> disjoint: OK')
split_of = {u: k for k, v in split_ids.items() for u in v}

{'train': 11790, 'val': 655, 'test': 655} -> disjoint: OK


## Step 2: Build the feature table

In [5]:
keep = set(qa.loc[~qa.exclude, 'utt_id'])
al = al[al.utt_id.isin(keep)].sort_values(['utt_id', 'phone_idx']).reset_index(drop=True)
g = al.groupby('utt_id')
al['next_is_sil'] = g['base'].shift(-1).eq('sil'); al['prev_is_sil'] = g['base'].shift(1).eq('sil')
al['is_speech'] = ~al.base.isin(['sil', 'spn'])
d = al[al.is_speech].copy().reset_index(drop=True)
d['dur_f'] = d['dur_f'].clip(lower=1)                           # a rounding artefact could give 0 frames
g = d.groupby('utt_id')
d['seq_idx'] = g.cumcount(); d['n_phones'] = g['base'].transform('size')
for k, name in [(1, 'prev1'), (2, 'prev2')]: d[name] = g['base'].shift(k).fillna('<s>')
for k, name in [(1, 'next1'), (2, 'next2')]: d[name] = g['base'].shift(-k).fillna('</s>')
d['rel_pos_utt'] = d['seq_idx'] / (d['n_phones'] - 1).clip(lower=1)
d['is_utt_first'] = (d['seq_idx'] == 0).astype(int); d['is_utt_final'] = (d['seq_idx'] == d['n_phones'] - 1).astype(int)
wg = d.groupby(['utt_id', 'word_idx'])
d['pos_in_word'] = wg.cumcount(); d['word_len'] = wg['base'].transform('size')
d['rel_pos_in_word'] = d['pos_in_word'] / (d['word_len'] - 1).clip(lower=1)
d['is_word_first'] = (d['pos_in_word'] == 0).astype(int); d['is_word_last'] = (d['pos_in_word'] == d['word_len'] - 1).astype(int)
d['n_words'] = g['word_idx'].transform('max') + 1
d['pause_after'] = d['next_is_sil'].astype(int); d['pause_before'] = d['prev_is_sil'].astype(int)
d['is_vowel'] = (d['base'].map(phone_class) == 'vowel').astype(int); d['cls'] = d['base'].map(phone_class)
d['stress_c'] = d['stress'].astype(str)
# punctuation that followed each word in the original text
pun = dict(zip(tr['id'], tr['puncts'].fillna('').str.split(',')))
mismatch = 0
def wp(u, w):
    global mismatch
    L = pun.get(u, [])
    return L[w] if 0 <= w < len(L) else 'n'
d['word_punct'] = [wp(u, w) for u, w in zip(d.utt_id, d.word_idx)]
d['split'] = d['utt_id'].map(split_of)
d['log_dur'] = np.log(d['dur_f'])
d['row_id'] = np.arange(len(d))
keep_cols = ['row_id', 'utt_id', 'split', 'phone_idx', 'dur_f', 'log_dur'] + CAT_COLS + [c for c in NUM_COLS if c not in CAT_COLS]
feat = d[keep_cols]
feat.to_csv(os.path.join(PROCESSED, 'features.csv.gz'), index=False)
print(feat.shape, '| rows per split:', feat.split.value_counts().to_dict())
feat.head()

(777055, 29) | rows per split: {'train': 700376, 'val': 38763, 'test': 37916}


,row_id,utt_id,split,phone_idx,dur_f,log_dur,base,stress_c,cls,prev1,...,word_idx,n_words,rel_pos_utt,seq_idx,n_phones,is_utt_final,is_utt_first,pause_after,pause_before,is_vowel
0,0,LJ001-0001,train,0,4,1.386294,P,-1,stop,<s>,...,0,27,0.000000,0,108,0,1,0,0,0
1,1,LJ001-0001,train,1,3,1.098612,R,-1,liquid,P,...,0,27,0.009346,1,108,0,0,0,0,0
2,2,LJ001-0001,train,2,6,1.791759,IH,1,vowel,R,...,0,27,0.018692,2,108,0,0,0,0,1
3,3,LJ001-0001,train,3,5,1.609438,N,-1,nasal,IH,...,0,27,0.028037,3,108,0,0,0,0,0
4,4,LJ001-0001,train,4,7,1.945910,T,-1,stop,N,...,0,27,0.037383,4,108,0,0,0,0,0


In [6]:
# ---- Sanity checks (what a reviewer would ask)
assert feat['dur_f'].min() >= 1 and feat.isna().sum().sum() == 0
print('missing values: none | min/max frames:', feat.dur_f.min(), feat.dur_f.max())
print('word-count agreement with transcript:', (tr.set_index('id').loc[feat.utt_id.unique(), 'n_words'].values == feat.groupby('utt_id', sort=False)['n_words'].first().values).mean().round(3), '(1.0 = every utterance matches)')
print(feat.groupby('split').dur_f.agg(['count', 'mean', 'std']).round(2))   # splits should look alike
print('stress values:', sorted(feat.stress_c.unique()), '| punct codes:', feat.word_punct.value_counts().to_dict())

missing values: none | min/max frames: 1 93
word-count agreement with transcript: 1.0 (1.0 = every utterance matches)
        count  mean   std
split                    
test    37916  7.48  4.11
train  700376  7.45  4.12
val     38763  7.49  4.14
stress values: ['-1', '0', '1', '2'] | punct codes: {'n': 663144, 'c': 71614, 'p': 42297}
